# FP32 vs INT8 (PTQ, QAT): test-set comparison

Loads the outputs of `evaluate_model.ipynb` for the three models and compares them: every metric with its change vs FP32, paired-bootstrap tests of the predictive differences, prediction agreement, and plots. Nothing is re-run here; all numbers come from `results/eval_<label>/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

LABELS = ["FP32", "INT8-PTQ", "INT8-QAT"]
COLORS = {"FP32": "#1f77b4", "INT8-PTQ": "#ff7f0e", "INT8-QAT": "#2ca02c"}
OUT_DIR = Path("results/comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)
N_BOOT = 2000

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.grid": True, "grid.color": "#e6e6e6", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "font.size": 9,
})

def save_fig(fig, name):
    fig.savefig(OUT_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

runs = {}
for label in LABELS:
    d = Path("results") / f"eval_{label}"
    runs[label] = {
        "metrics": json.load(open(d / "metrics.json")),
        "settings": json.load(open(d / "settings.json")),
        "pred": pd.read_csv(d / "predictions.csv"),
        "latency": pd.read_csv(d / "latency_samples.csv")["latency_ms"].to_numpy(),
    }
    s = runs[label]["settings"]
    print(f"{label:9s} {s['model_path']}  | evaluated {s['timestamp_utc']}  | threads {s['num_threads']}  | engine {s.get('quantized_engine')}")

## Same test images?

The comparison is refused unless all three runs scored exactly the same images, in the same order, with the same labels.

In [ ]:
ref = runs["FP32"]["pred"]
for label in LABELS[1:]:
    p = runs[label]["pred"]
    if not (len(p) == len(ref) and (p["image_id"].values == ref["image_id"].values).all()):
        raise RuntimeError(f"{label} was evaluated on different test images than FP32: refusing to compare")
    if not (p["true"].values == ref["true"].values).all():
        raise RuntimeError(f"{label} has different true labels than FP32: refusing to compare")
print(f"OK: all three runs scored the same {len(ref)} test images in the same order")

classes = [c[len("prob_"):] for c in ref.columns if c.startswith("prob_")]
cls_idx = {c: i for i, c in enumerate(classes)}
y_true = ref["true"].map(cls_idx).to_numpy()
y_pred = {l: runs[l]["pred"]["predicted"].map(cls_idx).to_numpy() for l in LABELS}
y_prob = {l: runs[l]["pred"][[f"prob_{c}" for c in classes]].to_numpy() for l in LABELS}

# The bootstrap below works from predictions.csv: check it reproduces each run's metrics.json exactly
for l in LABELS:
    perf = runs[l]["metrics"]["performance"]
    recomputed = {"accuracy": accuracy_score(y_true, y_pred[l]),
                  "macro_f1": f1_score(y_true, y_pred[l], average="macro", zero_division=0),
                  "macro_auc_ovr": roc_auc_score(y_true, y_prob[l], multi_class="ovr", average="macro",
                                                 labels=list(range(len(classes))))}
    for k, v in recomputed.items():
        if abs(v - perf[k]) > 1e-9:
            raise RuntimeError(f"{l}: {k} from predictions.csv ({v}) does not match metrics.json ({perf[k]})")
print("OK: accuracy, macro F1 and macro AUC recomputed from predictions.csv match metrics.json for all three runs")

## All metrics, with the change vs FP32

Predictive metrics change in percentage points (pp). Size, latency, throughput, memory and energy change in % (and as an x-fold ratio).

In [ ]:
def get(label, path):
    node = runs[label]["metrics"]
    for key in path:
        if node is None:
            return None
        node = node[key]
    return node

PREDICTIVE = [
    ("Accuracy", ("performance", "accuracy")),
    ("Balanced accuracy", ("performance", "balanced_accuracy")),
    ("Macro precision", ("performance", "macro_precision")),
    ("Macro recall", ("performance", "macro_recall")),
    ("Macro F1", ("performance", "macro_f1")),
    ("Macro specificity", ("performance", "macro_specificity")),
    ("Macro AUC (OvR)", ("performance", "macro_auc_ovr")),
]
PREDICTIVE += [(f"Recall {c}", ("per_class", c, "recall")) for c in classes]
PREDICTIVE += [(f"F1 {c}", ("per_class", c, "f1")) for c in classes]
COSTS = [  # (name, path, unit, lower_is_better)
    ("Model file size", ("efficiency", "model_size_mb"), "MB", True),
    ("GMACs", ("efficiency", "gmacs"), "GMACs", True),
    ("Latency bs=1, median", ("efficiency", "latency_bs1_ms", "median"), "ms", True),
    ("Latency bs=1, p95", ("efficiency", "latency_bs1_ms", "p95"), "ms", True),
    ("Throughput bs=32", ("efficiency", "throughput_bs32_ips", "images_per_s"), "img/s", False),
    ("Memory: model loaded", ("efficiency", "memory_mb", "bs1", "model_mb"), "MB", True),
    ("Memory: peak over baseline bs=1", ("efficiency", "memory_mb", "bs1", "peak_over_baseline_mb"), "MB", True),
    ("Memory: peak over baseline bs=32", ("efficiency", "memory_mb", "bs32", "peak_over_baseline_mb"), "MB", True),
    ("Energy CPU+RAM (mean)", ("energy", "wh_per_1000_images"), "Wh / 1,000 img", True),
    ("Energy CPU+RAM (sd)", ("energy", "wh_per_1000_images_std"), "Wh / 1,000 img", None),
    ("CO2e (mean)", ("energy", "g_co2e_per_1000_images"), "g / 1,000 img", True),
    ("CO2e (sd)", ("energy", "g_co2e_per_1000_images_std"), "g / 1,000 img", None),
]

rows = []
for name, path in PREDICTIVE:
    vals = {l: get(l, path) for l in LABELS}
    row = {"metric": name, "unit": "", **{l: vals[l] for l in LABELS}}
    for l in LABELS[1:]:
        row[f"change {l}"] = f"{(vals[l] - vals['FP32']) * 100:+.2f} pp"
    rows.append(row)
for name, path, unit, lower_better in COSTS:
    vals = {l: get(l, path) for l in LABELS}
    row = {"metric": name, "unit": unit, **{l: vals[l] for l in LABELS}}
    for l in LABELS[1:]:
        if vals[l] is None or vals["FP32"] is None or lower_better is None:  # sd rows: spread, not a comparison
            row[f"change {l}"] = ""
            continue
        pct = (vals[l] / vals["FP32"] - 1) * 100
        ratio = vals["FP32"] / vals[l] if lower_better else vals[l] / vals["FP32"]
        word = ("smaller" if "size" in name.lower() or "Memory" in name else "lower") if lower_better else "higher"
        row[f"change {l}"] = f"{pct:+.1f}% ({ratio:.2f}x {word})" if abs(pct) >= 0.05 else "0.0% (identical)"
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(OUT_DIR / "comparison_table.csv", index=False)
with pd.option_context("display.max_rows", None, "display.width", 200):
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nGMACs note for INT8:", get("INT8-PTQ", ("efficiency", "gmacs_note")))
if runs["FP32"]["metrics"]["energy"]:
    print("CPU power:", {l: get(l, ("energy", "cpu_power_source")) + " / " + get(l, ("energy", "cpu_power_mode")) for l in LABELS})

## Paired bootstrap of the differences vs FP32

Each of the 2,000 resamples draws the same test images for both models, so the interval reflects the difference itself, not the noise of each model separately. If 0 is inside the 95% CI, the difference is not distinguishable from chance.

In [ ]:
rng = np.random.default_rng(0)
boot_idx = rng.integers(0, len(y_true), size=(N_BOOT, len(y_true)))
labels_idx = list(range(len(classes)))

def metric_fns():
    return {
        "accuracy": lambda l, i: accuracy_score(y_true[i], y_pred[l][i]),
        "macro_f1": lambda l, i: f1_score(y_true[i], y_pred[l][i], average="macro", zero_division=0),
        "macro_auc_ovr": lambda l, i: roc_auc_score(y_true[i], y_prob[l][i], multi_class="ovr", average="macro", labels=labels_idx),
    }

boot_rows = []
for l in LABELS[1:]:
    for m, fn in metric_fns().items():
        diffs, skipped = [], 0
        for i in boot_idx:
            if m == "macro_auc_ovr" and len(np.unique(y_true[i])) < len(classes):
                skipped += 1  # AUC needs every class in the resample
                continue
            diffs.append(fn(l, i) - fn("FP32", i))
        full = np.arange(len(y_true))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        boot_rows.append({"comparison": f"{l} - FP32", "metric": m,
                          "observed_diff": fn(l, full) - fn("FP32", full),
                          "ci95_low": lo, "ci95_high": hi, "zero_in_ci": bool(lo <= 0 <= hi),
                          "resamples_used": len(diffs), "resamples_skipped": skipped})

boot = pd.DataFrame(boot_rows)
boot.to_csv(OUT_DIR / "paired_bootstrap.csv", index=False)
print(boot.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

## Prediction agreement with FP32

In [ ]:
agree_rows, class_rows = [], []
for l in LABELS[1:]:
    same = y_pred[l] == y_pred["FP32"]
    fp32_ok, q_ok = y_pred["FP32"] == y_true, y_pred[l] == y_true
    agree_rows.append({"model": l, "identical_to_fp32_pct": same.mean() * 100, "changed": int((~same).sum()),
                       "fixed (FP32 wrong -> correct)": int((~fp32_ok & q_ok).sum()),
                       "broken (FP32 correct -> wrong)": int((fp32_ok & ~q_ok).sum()),
                       "wrong -> other wrong": int((~fp32_ok & ~q_ok & ~same).sum())})
for c in classes:
    mask = y_true == cls_idx[c]
    class_rows.append({"true class": c, "test images": int(mask.sum()),
                       **{f"changed {l}": int((y_pred[l][mask] != y_pred["FP32"][mask]).sum()) for l in LABELS[1:]}})

agreement = pd.DataFrame(agree_rows)
per_class_changes = pd.DataFrame(class_rows)
agreement.to_csv(OUT_DIR / "prediction_agreement.csv", index=False)
per_class_changes.to_csv(OUT_DIR / "changed_predictions_per_class.csv", index=False)
print(agreement.round(2).to_string(index=False))
print()
print(per_class_changes.to_string(index=False))

## Plots

In [ ]:
# 1. Predictive metrics, grouped, with 95% bootstrap CIs where computed
pm = [("accuracy", "Accuracy"), ("balanced_accuracy", "Balanced\naccuracy"), ("macro_precision", "Macro\nprecision"),
      ("macro_recall", "Macro\nrecall"), ("macro_f1", "Macro F1"), ("macro_specificity", "Macro\nspecificity"),
      ("macro_auc_ovr", "Macro AUC\n(OvR)")]
x = np.arange(len(pm))
w = 0.26
fig, ax = plt.subplots(figsize=(9, 3.6))
for k, l in enumerate(LABELS):
    perf = runs[l]["metrics"]["performance"]
    vals = np.array([perf[m] for m, _ in pm])
    ci = perf["ci95"]
    err = np.array([[perf[m] - ci[m][0], ci[m][1] - perf[m]] if m in ci else [np.nan, np.nan] for m, _ in pm]).T
    pos = x + (k - 1) * w
    ax.bar(pos, vals, w * 0.92, color=COLORS[l], label=l)
    has = ~np.isnan(err[0])
    ax.errorbar(pos[has], vals[has], yerr=err[:, has], fmt="none", ecolor="black", elinewidth=0.8, capsize=2)
ax.set_xticks(x)
ax.set_xticklabels([n for _, n in pm])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title(f"Test metrics (n={len(y_true)}); error bars: 95% bootstrap CI where computed")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="x", visible=False)
save_fig(fig, "1_predictive_metrics")
plt.show()

In [ ]:
# 2. Per-class recall
xc = np.arange(len(classes))
fig, ax = plt.subplots(figsize=(9, 3.4))
support = [runs["FP32"]["metrics"]["per_class"][c]["support"] for c in classes]
for k, l in enumerate(LABELS):
    ax.bar(xc + (k - 1) * w, [runs[l]["metrics"]["per_class"][c]["recall"] for c in classes], w * 0.92,
           color=COLORS[l], label=l)
ax.set_xticks(xc)
ax.set_xticklabels([f"{c}\n(n={n})" for c, n in zip(classes, support)])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Recall (sensitivity)")
ax.set_title("Per-class recall on the test set")
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.grid(axis="x", visible=False)
save_fig(fig, "2_per_class_recall")
plt.show()

In [ ]:
# 3. Costs: size, latency (box plots of the saved samples), throughput, memory, energy, CO2e
has_energy = all(runs[l]["metrics"]["energy"] for l in LABELS)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
axes = axes.ravel()
colors = [COLORS[l] for l in LABELS]
short = ["FP32", "PTQ", "QAT"]

def bars(ax, values, title, ylabel, fmt, err=None):
    ax.bar(short, values, color=colors, width=0.6)
    if err is not None:
        ax.errorbar(range(len(values)), values, yerr=err, fmt="none", ecolor="black", elinewidth=1, capsize=3)
    top = [v + (e if err is not None else 0) for v, e in zip(values, err if err is not None else [0] * len(values))]
    for i, (v, t) in enumerate(zip(values, top)):
        ax.text(i, t, fmt.format(v), ha="center", va="bottom", fontsize=7.5)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(top) * 1.18)
    ax.grid(axis="x", visible=False)

bars(axes[0], [get(l, ("efficiency", "model_size_mb")) for l in LABELS], "Model file size", "MB", "{:.1f}")
bp = axes[1].boxplot([runs[l]["latency"] for l in LABELS], tick_labels=short, patch_artist=True, widths=0.55,
                     medianprops={"color": "black"}, flierprops={"markersize": 2})
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
axes[1].set_title(f"Latency, batch 1 ({len(runs['FP32']['latency'])} runs)")
axes[1].set_ylabel("ms per image")
axes[1].set_ylim(bottom=0)
axes[1].grid(axis="x", visible=False)
bars(axes[2], [get(l, ("efficiency", "throughput_bs32_ips", "images_per_s")) for l in LABELS],
     "Throughput, batch 32", "images / s", "{:.0f}")
bars(axes[3], [get(l, ("efficiency", "memory_mb", "bs1", "model_mb")) for l in LABELS],
     "Memory: model loaded", "MB above baseline", "{:.0f}")
bars(axes[4], [get(l, ("efficiency", "memory_mb", "bs1", "peak_over_baseline_mb")) for l in LABELS],
     "Memory: peak, batch 1", "MB above baseline", "{:.0f}")
bars(axes[5], [get(l, ("efficiency", "memory_mb", "bs32", "peak_over_baseline_mb")) for l in LABELS],
     "Memory: peak, batch 32", "MB above baseline", "{:.0f}")
if has_energy:
    bars(axes[6], [get(l, ("energy", "wh_per_1000_images")) for l in LABELS],
         "Energy, CPU + RAM (mean ± sd)", "Wh per 1,000 images", "{:.3f}",
         err=[get(l, ("energy", "wh_per_1000_images_std")) for l in LABELS])
    bars(axes[7], [get(l, ("energy", "g_co2e_per_1000_images")) for l in LABELS],
         "CO₂e (mean ± sd)", "g per 1,000 images", "{:.3f}",
         err=[get(l, ("energy", "g_co2e_per_1000_images_std")) for l in LABELS])
else:
    for ax in axes[6:]:
        ax.set_visible(False)
fig.suptitle("Memory measured in a fresh process per model; energy repeated "
             f"{get('FP32', ('energy', 'n_repeats')) if has_energy else 0}x per model", fontsize=9, y=1.0)
fig.tight_layout()
save_fig(fig, "3_costs")
plt.show()

In [ ]:
# 4. Trade-offs: macro F1 vs median latency and vs energy
fig, axes = plt.subplots(1, 2 if has_energy else 1, figsize=(9.5 if has_energy else 4.8, 3.6), squeeze=False)
axes = axes[0]
label_offset = {"FP32": (8, 6), "INT8-PTQ": (10, 8), "INT8-QAT": (10, -14)}  # PTQ and QAT sit at almost the same x
for l in LABELS:
    perf = runs[l]["metrics"]["performance"]
    f1, (lo, hi) = perf["macro_f1"], perf["ci95"]["macro_f1"]
    lat = runs[l]["latency"]
    med, q1, q3 = np.median(lat), np.percentile(lat, 25), np.percentile(lat, 75)
    axes[0].errorbar(med, f1, yerr=[[f1 - lo], [hi - f1]], xerr=[[med - q1], [q3 - med]], fmt="o",
                     color=COLORS[l], ecolor=COLORS[l], elinewidth=1, capsize=3, markersize=7, label=l)
    axes[0].annotate(l, (med, f1), textcoords="offset points", xytext=label_offset[l], fontsize=8, color=COLORS[l])
    if has_energy:
        e, e_sd = get(l, ("energy", "wh_per_1000_images")), get(l, ("energy", "wh_per_1000_images_std"))
        axes[1].errorbar(e, f1, yerr=[[f1 - lo], [hi - f1]], xerr=[[e_sd], [e_sd]], fmt="o", color=COLORS[l],
                         ecolor=COLORS[l], elinewidth=1, capsize=3, markersize=7, label=l)
        axes[1].annotate(l, (e, f1), textcoords="offset points", xytext=label_offset[l], fontsize=8, color=COLORS[l])
axes[0].set_xlabel("Median latency, batch 1 (ms); x bars: interquartile range")
axes[0].set_ylabel("Macro F1 (each model's 95% bootstrap CI)")
axes[0].set_title("Macro F1 vs latency")
axes[0].set_xlim(left=0)
if has_energy:
    axes[1].set_xlabel("Energy, CPU + RAM (Wh per 1,000 images); x bars: ± sd")
    axes[1].set_title("Macro F1 vs energy")
    axes[1].set_xlim(left=0)
fig.tight_layout()
save_fig(fig, "4_tradeoffs")
plt.show()

In [ ]:
# 5. Paired differences vs FP32: the right interval for "did quantization change the score?"
# Each model's own CI (plots 1 and 4) overlaps the others' even when the paired difference is clearly non-zero,
# so the comparison is judged here, on the paired bootstrap.
names = {"accuracy": "Accuracy", "macro_f1": "Macro F1", "macro_auc_ovr": "Macro AUC (OvR)"}
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ypos = []
for j, m in enumerate(names):
    for k, l in enumerate(LABELS[1:]):
        r = boot[(boot["comparison"] == f"{l} - FP32") & (boot["metric"] == m)].iloc[0]
        y = j * 3 + k
        ypos.append((y, f"{names[m]}  {l.split('-')[1]}"))
        ax.errorbar(r["observed_diff"] * 100, y, xerr=[[(r["observed_diff"] - r["ci95_low"]) * 100],
                                                     [(r["ci95_high"] - r["observed_diff"]) * 100]],
                    fmt="o", color=COLORS[l], ecolor=COLORS[l], elinewidth=1.5, capsize=3, markersize=6)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks([y for y, _ in ypos])
ax.set_yticklabels([t for _, t in ypos])
ax.invert_yaxis()
ax.set_xlabel("Difference vs FP32 (percentage points), 95% paired bootstrap CI")
ax.set_title(f"INT8 minus FP32 on the same {len(y_true)} test images ({N_BOOT} paired resamples)")
ax.grid(axis="y", visible=False)
save_fig(fig, "5_paired_differences")
plt.show()

print("Saved to", OUT_DIR, ":", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))